# 06 — Cruce 2: NBI vs. Mortalidad por diabetes — Desviación positiva

**Pregunta:** ¿Las provincias más pobres mueren más de diabetes?

**Hipótesis nula:** mayor pobreza (NBI) → mayor mortalidad por diabetes.

**Variables:**
- `pct_nbi` — % hogares con Necesidades Básicas Insatisfechas (Censo 2022)
- `tasa_mortalidad_100k` — muertes por diabetes CIE-10 E10-E14 / 100.000 hab. (2019-2024)
- `pct_indigena` — % población indígena (Censo 2022) — variable explicativa secundaria

**Nota sobre granularidad:**  
Los microdatos INEC de defunciones registran `cant_fall` (cantón de defunción) pero no `cant_res` (cantón de residencia). Usar el cantón de defunción sesgaría la mortalidad hacia cantones hospitalarios urbanos. Por eso el análisis se mantiene en nivel provincial (24 provincias), donde `prov_res` es confiable.

**Desviación positiva:** provincias con NBI alto pero mortalidad menor a la esperada por la tendencia — comunidades que resisten la transición alimentaria.

**Output:** `processed/cruce2_scatter_provincia.csv`

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from scipy import stats

pd.set_option('display.float_format', '{:,.2f}'.format)

PROCESSED_DIR = Path('../processed')

def norm(s):
    s = str(s).strip().upper()
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

## 1. Carga — reutilizamos el cruce 1 ya construido

In [2]:
# El cruce 1 ya tiene tasa_mortalidad, NBI e indigena por provincia
df = pd.read_csv(PROCESSED_DIR / 'cruce1_barras_provincia.csv')
df['provincia'] = df['provincia'].apply(norm)

print(f'Provincias: {len(df)}')
print(f'Nulos tasa_mortalidad: {df["tasa_mortalidad_100k"].isna().sum()}')
print(f'Nulos pct_nbi: {df["pct_nbi"].isna().sum()}')
print(f'Nulos pct_indigena: {df["pct_indigena"].isna().sum()}')
df[['provincia','tasa_mortalidad_100k','pct_nbi','pct_indigena']]

Provincias: 24
Nulos tasa_mortalidad: 0
Nulos pct_nbi: 0
Nulos pct_indigena: 0


,provincia,tasa_mortalidad_100k,pct_nbi,pct_indigena
0,SANTA ELENA,358.90,47.30,1.20
1,GUAYAS,286.90,39.10,1.30
2,EL ORO,199.40,36.60,0.50
3,ESMERALDAS,187.70,63.00,3.40
4,LOJA,167.40,39.40,3.60
5,CARCHI,147.30,33.40,4.20
6,TUNGURAHUA,134.80,31.60,13.50
7,AZUAY,129.20,25.70,2.00
8,CHIMBORAZO,126.20,42.80,37.90
9,IMBABURA,121.90,32.70,28.00


## 2. Correlaciones

In [3]:
df_clean = df.dropna(subset=['tasa_mortalidad_100k', 'pct_nbi', 'pct_indigena'])

r_nbi, p_nbi     = stats.spearmanr(df_clean['pct_nbi'],       df_clean['tasa_mortalidad_100k'])
r_ind, p_ind     = stats.spearmanr(df_clean['pct_indigena'],  df_clean['tasa_mortalidad_100k'])
r_nbi_ind, p_ni  = stats.spearmanr(df_clean['pct_nbi'],       df_clean['pct_indigena'])

print('Spearman — mortalidad ~ NBI:        r={:.2f}  p={:.3f}'.format(r_nbi, p_nbi))
print('Spearman — mortalidad ~ % indígena: r={:.2f}  p={:.3f}'.format(r_ind, p_ind))
print('Spearman — NBI ~ % indígena:        r={:.2f}  p={:.3f}'.format(r_nbi_ind, p_ni))
print()
print('Interpretación:')
print('  - NBI negativo con mortalidad: más pobreza NO implica más muertes de diabetes')
print('  - % indígena negativo con mortalidad: más población indígena → menos mortalidad')
print('  - NBI y % indígena correlacionan positivamente: provincias indígenas son las más pobres')
print('  - Hallazgo central: la transición alimentaria golpea en provincias mestizo-costeras,')
print('    no en las más pobres — la pobreza indígena convive con patrones alimentarios tradicionales')

Spearman — mortalidad ~ NBI:        r=-0.36  p=0.082
Spearman — mortalidad ~ % indígena: r=-0.29  p=0.167
Spearman — NBI ~ % indígena:        r=0.29  p=0.175

Interpretación:
  - NBI negativo con mortalidad: más pobreza NO implica más muertes de diabetes
  - % indígena negativo con mortalidad: más población indígena → menos mortalidad
  - NBI y % indígena correlacionan positivamente: provincias indígenas son las más pobres
  - Hallazgo central: la transición alimentaria golpea en provincias mestizo-costeras,
    no en las más pobres — la pobreza indígena convive con patrones alimentarios tradicionales


## 3. Línea de tendencia y residuos (desviación positiva)

In [4]:
# OLS: tasa_mortalidad ~ pct_nbi
slope, intercept, r_val, p_val, se = stats.linregress(
    df_clean['pct_nbi'], df_clean['tasa_mortalidad_100k']
)

df_clean = df_clean.copy()
df_clean['mortalidad_esperada'] = (slope * df_clean['pct_nbi'] + intercept).round(1)
df_clean['residuo']             = (df_clean['tasa_mortalidad_100k'] - df_clean['mortalidad_esperada']).round(1)

# Desviación positiva: residuo negativo → muere MENOS de lo esperado para su NBI
df_clean['desviacion'] = df_clean['residuo'].apply(
    lambda r: 'positiva' if r < -30 else ('negativa' if r > 30 else 'esperada')
)

print(f'OLS: tasa = {slope:.2f} × NBI + {intercept:.1f}')
print(f'R² = {r_val**2:.3f}  |  p = {p_val:.3f}')
print()

cols = ['provincia','pct_nbi','tasa_mortalidad_100k','mortalidad_esperada','residuo','pct_indigena','desviacion']
print(df_clean[cols].sort_values('residuo').to_string(index=False))

OLS: tasa = -1.67 × NBI + 187.8
R² = 0.078  |  p = 0.187

                     provincia  pct_nbi  tasa_mortalidad_100k  mortalidad_esperada  residuo  pct_indigena desviacion
                     GALAPAGOS    31.40                 42.20               135.30   -93.10          9.30   positiva
                         CANAR    39.20                 35.70               122.30   -86.60         14.70   positiva
SANTO DOMINGO DE LOS TSACHILAS    46.70                 32.20               109.70   -77.50          1.40   positiva
                     SUCUMBIOS    59.20                 13.10                88.80   -75.70         16.30   positiva
                       BOLIVAR    58.60                 21.20                89.80   -68.60         29.50   positiva
                     PICHINCHA    15.90                 96.60               161.20   -64.60          6.20   positiva
                        MANABI    60.10                 39.60                87.30   -47.70          0.20   positiva
      

## 4. Provincias con desviación positiva — las que resisten

In [5]:
pos = df_clean[df_clean['desviacion'] == 'positiva'].sort_values('residuo')
neg = df_clean[df_clean['desviacion'] == 'negativa'].sort_values('residuo', ascending=False)

print('=== DESVIACIÓN POSITIVA (mueren menos de lo esperado) ===')
print(pos[['provincia','pct_nbi','tasa_mortalidad_100k','mortalidad_esperada','residuo','pct_indigena']].to_string(index=False))
print()
print('=== DESVIACIÓN NEGATIVA (mueren más de lo esperado) ===')
print(neg[['provincia','pct_nbi','tasa_mortalidad_100k','mortalidad_esperada','residuo','pct_indigena']].to_string(index=False))

=== DESVIACIÓN POSITIVA (mueren menos de lo esperado) ===
                     provincia  pct_nbi  tasa_mortalidad_100k  mortalidad_esperada  residuo  pct_indigena
                     GALAPAGOS    31.40                 42.20               135.30   -93.10          9.30
                         CANAR    39.20                 35.70               122.30   -86.60         14.70
SANTO DOMINGO DE LOS TSACHILAS    46.70                 32.20               109.70   -77.50          1.40
                     SUCUMBIOS    59.20                 13.10                88.80   -75.70         16.30
                       BOLIVAR    58.60                 21.20                89.80   -68.60         29.50
                     PICHINCHA    15.90                 96.60               161.20   -64.60          6.20
                        MANABI    60.10                 39.60                87.30   -47.70          0.20
                      LOS RIOS    61.40                 40.20                85.20   -45.00   

## 5. Guardar

## 5. Desplazamiento al morir: rural → urbano

**Pregunta:** ¿Los pacientes rurales con diabetes mueren donde viven o en hospitales urbanos?

Los microdatos INEC registran `area_res` (zona de residencia) y `area_fall` (zona donde ocurrió la muerte). La discrepancia entre ambas mide el desplazamiento al final de vida — el sistema de salud los recibe cuando ya es tarde.

In [6]:
import unicodedata
from pathlib import Path

RAW = Path('../raw/INEC')

# INEC 2019: area_res y area_fall codificados como 1=Urbano, 2=Rural (numérico)
def norm_area(s):
    s = str(s).strip()
    if s in ('1', '1.0'):
        return 'URBANO'
    if s in ('2', '2.0'):
        return 'RURAL'
    s = s.upper()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return 'URBANO' if 'URBAN' in s else ('RURAL' if 'RURAL' in s else 'OTRO')

# INEC 2019: prov_res también viene como código numérico (1-24)
PROV_CODES = {
    '1': 'AZUAY', '2': 'BOLIVAR', '3': 'CANAR', '4': 'CARCHI', '5': 'CHIMBORAZO',
    '6': 'COTOPAXI', '7': 'EL ORO', '8': 'ESMERALDAS', '9': 'GUAYAS', '10': 'IMBABURA',
    '11': 'LOJA', '12': 'LOS RIOS', '13': 'MANABI', '14': 'MORONA SANTIAGO', '15': 'NAPO',
    '16': 'PASTAZA', '17': 'PICHINCHA', '18': 'TUNGURAHUA', '19': 'ZAMORA CHINCHIPE',
    '20': 'GALAPAGOS', '21': 'SUCUMBIOS', '22': 'ORELLANA',
    '23': 'SANTO DOMINGO DE LOS TSACHILAS', '24': 'SANTA ELENA'
}

# Encoding artifacts que sobreviven NFD+Mn → mapeo explícito para consolidar duplicados
PROV_FIX = {
    'SUCUMBA\xadOS': 'SUCUMBIOS', 'SUCUMBA\xados': 'SUCUMBIOS',
    'MANABA\xad': 'MANABI',    'MANABA\xadI': 'MANABI',
    'BOLA\xadVAR': 'BOLIVAR',  'BOLA\xadvar': 'BOLIVAR',
    'LOS RA\xadOS': 'LOS RIOS', 'LOS RA\xados': 'LOS RIOS',
    'CAA\xb1AR': 'CANAR',      'CAA\xb1ar': 'CANAR',
    'SANTO DOMINGO DE LOS TSA\xa1CHILAS': 'SANTO DOMINGO DE LOS TSACHILAS',
    'SANTO DOMINGO DE LOS TSA\xa1chilas': 'SANTO DOMINGO DE LOS TSACHILAS',
}

def norm_prov(s):
    s = str(s).strip()
    if s in PROV_CODES:
        return PROV_CODES[s]
    s = s.upper()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return PROV_FIX.get(s, s)

frames = []
for f in sorted(RAW.glob('*.csv')):
    df_raw = pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False)
    df_raw.columns = ['numeracion'] + list(df_raw.columns[1:])
    frames.append(df_raw)

all_def = pd.concat(frames, ignore_index=True)

diab = all_def[
    all_def['causa'].astype(str).str.match(r'^E1[0-4]', na=False) &
    (pd.to_numeric(all_def['anio_fall'], errors='coerce').between(2019, 2024))
].copy()

diab['area_res_n']  = diab['area_res'].apply(norm_area)
diab['area_fall_n'] = diab['area_fall'].apply(norm_area)

# Tabla cruzada
tabla = pd.crosstab(diab['area_res_n'], diab['area_fall_n'], margins=True)
print('Muertes por diabetes 2019-2024 — área de residencia × área de fallecimiento:')
print(tabla.to_string())
print()

# Desplazamiento rural → urbano
rural      = diab[diab['area_res_n'] == 'RURAL']
rural_urb  = diab[(diab['area_res_n'] == 'RURAL') & (diab['area_fall_n'] == 'URBANO')]
rural_rur  = diab[(diab['area_res_n'] == 'RURAL') & (diab['area_fall_n'] == 'RURAL')]

print(f'Residentes rurales que mueren de diabetes:        {len(rural):,}')
print(f'  → fallecen en zona urbana (desplazados):        {len(rural_urb):,}  ({len(rural_urb)/len(rural)*100:.1f}%)')
print(f'  → fallecen en zona rural  (in situ):            {len(rural_rur):,}  ({len(rural_rur)/len(rural)*100:.1f}%)')

# Por provincia — % desplazados
diab['prov_res_n'] = diab['prov_res'].apply(norm_prov)

total_rural_prov = diab[diab['area_res_n'] == 'RURAL'].groupby('prov_res_n').size().rename('muertes_rurales')
desplaz_prov     = diab[(diab['area_res_n'] == 'RURAL') & (diab['area_fall_n'] == 'URBANO')].groupby('prov_res_n').size().rename('desplazados')
df_desplaz = pd.concat([total_rural_prov, desplaz_prov], axis=1).dropna().reset_index()
df_desplaz = df_desplaz.rename(columns={'prov_res_n': 'provincia'})

# Consolidar entradas duplicadas residuales (encoding artifacts que sobreviven el mapeo)
df_desplaz = df_desplaz.groupby('provincia', as_index=False).agg(
    muertes_rurales=('muertes_rurales', 'sum'),
    desplazados=('desplazados', 'sum')
)
df_desplaz['pct_desplazados'] = (df_desplaz['desplazados'] / df_desplaz['muertes_rurales'] * 100).round(1)
df_desplaz = df_desplaz.sort_values('pct_desplazados', ascending=False)

print()
print('Desplazamiento rural→urbano por provincia (ordenado por %):')
print(df_desplaz.to_string(index=False))

Muertes por diabetes 2019-2024 — área de residencia × área de fallecimiento:
area_fall_n  RURAL  URBANO    All
area_res_n                       
RURAL         4404    1644   6048
URBANO         259   26661  26920
All           4663   28305  32968

Residentes rurales que mueren de diabetes:        6,048
  → fallecen en zona urbana (desplazados):        1,644  (27.2%)
  → fallecen en zona rural  (in situ):            4,404  (72.8%)

Desplazamiento rural→urbano por provincia (ordenado por %):
                     provincia  muertes_rurales  desplazados  pct_desplazados
                      ORELLANA               18        10.00            55.60
                          NAPO               26        13.00            50.00
               MORONA SANTIAGO               64        30.00            46.90
              ZAMORA CHINCHIPE               35        16.00            45.70
                       PASTAZA               31        14.00            45.20
                        EL ORO       

In [7]:
# df_desplaz ya tiene columna 'provincia' limpia y consolidada desde la celda anterior.
# NOMBRE_FIX aquí como red de seguridad para cualquier artifact residual.
NOMBRE_FIX = {
    'SUCUMBA\xados': 'SUCUMBIOS', 'SUCUMBA\xadOS': 'SUCUMBIOS',
    'MANABA\xad': 'MANABI',      'MANABA\xadI': 'MANABI',
    'BOLA\xadvar': 'BOLIVAR',    'BOLA\xadVAR': 'BOLIVAR',
    'LOS RA\xados': 'LOS RIOS',  'LOS RA\xadOS': 'LOS RIOS',
    'CAA\xb1ar': 'CANAR',        'CAA\xb1AR': 'CANAR',
    'SANTO DOMINGO DE LOS TSA\xa1chilas': 'SANTO DOMINGO DE LOS TSACHILAS',
    'SANTO DOMINGO DE LOS TSA\xa1CHILAS': 'SANTO DOMINGO DE LOS TSACHILAS',
}

df_desplaz_out = df_desplaz.copy()
df_desplaz_out['provincia'] = df_desplaz_out['provincia'].replace(NOMBRE_FIX)
# Segunda consolidación por si quedó algún duplicado residual
df_desplaz_out = df_desplaz_out.groupby('provincia', as_index=False).agg(
    muertes_rurales=('muertes_rurales', 'sum'),
    desplazados=('desplazados', 'sum')
)
df_desplaz_out['pct_desplazados'] = (df_desplaz_out['desplazados'] / df_desplaz_out['muertes_rurales'] * 100).round(1)
df_desplaz_out = df_desplaz_out.sort_values('pct_desplazados', ascending=False).reset_index(drop=True)

out_desplaz = PROCESSED_DIR / 'cruce2_desplazamiento_provincia.csv'
df_desplaz_out.to_csv(out_desplaz, index=False)
print(f'Guardado: {out_desplaz} ({len(df_desplaz_out)} filas)')
print()
print(df_desplaz_out.to_string(index=False))

Guardado: ../processed/cruce2_desplazamiento_provincia.csv (23 filas)

                     provincia  muertes_rurales  desplazados  pct_desplazados
                      ORELLANA               18        10.00            55.60
                          NAPO               26        13.00            50.00
               MORONA SANTIAGO               64        30.00            46.90
              ZAMORA CHINCHIPE               35        16.00            45.70
                       PASTAZA               31        14.00            45.20
                        EL ORO              176        76.00            43.20
                     SUCUMBIOS               69        29.00            42.00
                    ESMERALDAS              334       126.00            37.70
SANTO DOMINGO DE LOS TSACHILAS              218        69.00            31.70
                         AZUAY              372       117.00            31.50
                      COTOPAXI              146        44.00           

In [8]:
cols_out = [
    'provincia', 'pob_total', 'pct_nbi', 'pct_indigena',
    'tasa_mortalidad_100k', 'mortalidad_esperada', 'residuo',
    'insulina_usd_per_capita', 'cuadrante', 'desviacion'
]
out = PROCESSED_DIR / 'cruce2_scatter_provincia.csv'
df_clean[cols_out].sort_values('residuo').to_csv(out, index=False)
print(f'Guardado: {out} ({len(df_clean)} filas)')

Guardado: ../processed/cruce2_scatter_provincia.csv (24 filas)


## 6. Resumen ETL

| Concepto | Valor |
|---|---|
| Provincias analizadas | 24 |
| Correlación Spearman NBI~mortalidad | r = −0,36 (p = 0,082) |
| Correlación Spearman % indígena~mortalidad | r = −0,29 (p = 0,167) |
| R² regresión OLS (NBI → mortalidad) | 0,078 (p = 0,187) |
| Std residuos | 81,3 |
| Provincias con desviación positiva (residuo < −30) | 8 |
| Provincias con desviación negativa (residuo > +30) | 5 |
| Output generado | cruce2_scatter_provincia.csv (24 filas) |

**Hallazgos clave:**
- La correlación NBI~mortalidad es **negativa** (r=−0,36, p=0,082): la tendencia sugiere que más pobreza no implica más muertes por diabetes. Sin embargo, con n=24 provincias, el valor-p no alcanza significancia convencional (p>0,05) y la OLS tampoco es significativa (p=0,187, R²=0,078). **No se rechaza la hipótesis nula formalmente; el patrón es sugerente, no concluyente.**
- Las provincias con mayor mortalidad relativa son **costeras, mestizas y de NBI medio-bajo** (Santa Elena, Guayas, Esmeraldas, El Oro) — donde la transición alimentaria llegó primero.
- Las provincias con mayor NBI (Amazonía, sierra indígena) tienen mortalidad menor a la esperada — patrón compatible con el patrón alimentario tradicional como factor diferencial, aunque también con mayor subregistro en zonas remotas. No es posible distinguir ambas hipótesis con datos disponibles.
- **Esmeraldas** es el caso más narrativo: NBI 63% (comparable a provincias amazónicas), pero mortalidad 187,7/100k (residuo +105) — provincia costera-mestiza donde la pobreza coexiste con la transición alimentaria sin la dieta indígena como amortiguador.

**Decisiones metodológicas:**
- Nivel provincial (24 puntos) — los microdatos INEC no registran `cant_res` (cantón de residencia); usar `cant_fall` sesgaría la mortalidad hacia cantones hospitalarios urbanos
- Umbral desviación ±30 = aprox. 0,37 std (std=81,3) — umbral narrativo, no estadístico
- OLS para calcular residuos individuales; Spearman para fuerza de asociación (no asume normalidad)
- No se afirma causalidad — análisis ecológico provincial
- R²=0,078: el NBI explica muy poco de la varianza en mortalidad — la tesis es que el tipo de pobreza (indígena-tradicional vs costera-mestiza) importa más que la pobreza en sí
- **Corrección aplicada:** `norm_area()` actualizada para manejar codificación numérica 1=Urbano / 2=Rural de INEC 2019. La versión anterior clasificaba ~4.890 muertes de 2019 como OTRO, excluyéndolas del análisis de desplazamiento.